# Benchmark Comparison: RL vs LLM vs Heuristic Agents

Compare joint HPA+VPA autoscaling across all agent types.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

from matplotlib import rc
rc('text', usetex=True)
rc('font', **{'family': 'serif', 'serif': ['Times New Roman']})

def moving_average(data, window_size):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

In [ ]:
# Load benchmark results
results_dir = '../results/benchmark/small'

algorithm_data = {}
for filepath in sorted(glob.glob(os.path.join(results_dir, '*_iter*_metrics.csv'))):
    filename = os.path.basename(filepath)
    alg_name = filename.rsplit('_iter', 1)[0]
    if alg_name not in algorithm_data:
        algorithm_data[alg_name] = []
    algorithm_data[alg_name].append(pd.read_csv(filepath))

print(f'Loaded algorithms: {list(algorithm_data.keys())}')

In [ ]:
# Response Time Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

colors = plt.cm.tab10(np.linspace(0, 1, len(algorithm_data)))

# Mean RT per algorithm
alg_names = []
mean_rts = []
p95_rts = []
for alg_name, dfs in algorithm_data.items():
    combined = pd.concat(dfs)
    alg_names.append(alg_name)
    mean_rts.append(combined['response_time'].mean())
    p95_rts.append(combined['response_time'].quantile(0.95))

x = np.arange(len(alg_names))
axes[0].bar(x - 0.2, mean_rts, 0.4, label='Mean', color=colors[:len(alg_names)])
axes[0].bar(x + 0.2, p95_rts, 0.4, label='P95', alpha=0.7, color=colors[:len(alg_names)])
axes[0].set_xticks(x)
axes[0].set_xticklabels(alg_names, rotation=45, ha='right')
axes[0].set_ylabel('Response Time (s)')
axes[0].set_title('Response Time by Algorithm')
axes[0].legend()
axes[0].axhline(y=0.250, color='r', linestyle='--', alpha=0.5, label='SLA')

# SLA violations
sla_violations = []
for alg_name, dfs in algorithm_data.items():
    combined = pd.concat(dfs)
    violations = (combined['response_time'] > 0.250).sum() / len(combined) * 100
    sla_violations.append(violations)

axes[1].bar(alg_names, sla_violations, color=colors[:len(alg_names)])
axes[1].set_ylabel('SLA Violations (\%)')
axes[1].set_title('SLA Violation Rate (>250ms)')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# CPU Utilization Over Time (per phase)
fig, axes = plt.subplots(len(algorithm_data), 1, figsize=(14, 4 * len(algorithm_data)), dpi=200, sharex=True)
if len(algorithm_data) == 1:
    axes = [axes]

for idx, (alg_name, dfs) in enumerate(algorithm_data.items()):
    combined = pd.concat(dfs)
    # Average across iterations per step
    step_avg = combined.groupby('step').agg({'cpu_percentage': 'mean', 'phase': 'first'}).reset_index()
    
    axes[idx].plot(step_avg['step'], step_avg['cpu_percentage'], linewidth=0.8)
    axes[idx].axhline(y=60, color='r', linestyle='--', alpha=0.3, label='Upper target (60\%)')
    axes[idx].axhline(y=30, color='g', linestyle='--', alpha=0.3, label='Lower target (30\%)')
    axes[idx].set_ylabel('CPU \%')
    axes[idx].set_title(f'{alg_name}')
    axes[idx].legend(loc='upper right', fontsize=8)

    # Shade phases
    phases = step_avg.groupby('phase')['step'].agg(['min', 'max'])
    for phase_name, (start, end) in phases.iterrows():
        axes[idx].axvspan(start, end, alpha=0.1)
        axes[idx].text((start+end)/2, axes[idx].get_ylim()[1]*0.95, phase_name,
                       ha='center', fontsize=7)

axes[-1].set_xlabel('Step')
plt.tight_layout()
plt.show()

In [ ]:
# Decision Latency Comparison
fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

latency_data = []
labels = []
for alg_name, dfs in algorithm_data.items():
    combined = pd.concat(dfs)
    latency_data.append(combined['decision_latency_ms'].values)
    labels.append(alg_name)

bp = ax.boxplot(latency_data, labels=labels, showfliers=False)
ax.set_ylabel('Decision Latency (ms)')
ax.set_title('Agent Decision Latency Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Cumulative Reward Comparison
fig, ax = plt.subplots(figsize=(12, 5), dpi=300)

for alg_name, dfs in algorithm_data.items():
    combined = pd.concat(dfs)
    step_reward = combined.groupby('step')['reward'].mean()
    cumulative = step_reward.cumsum()
    ax.plot(cumulative.index, cumulative.values, label=alg_name, linewidth=1.5)

ax.set_xlabel('Step')
ax.set_ylabel('Cumulative Reward')
ax.set_title('Cumulative Reward Over Time')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Resource Efficiency: CPU allocated vs used
fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

efficiency = []
for alg_name, dfs in algorithm_data.items():
    combined = pd.concat(dfs)
    # Efficiency = mean utilization (closer to target range is better)
    mean_util = combined['cpu_percentage'].mean()
    in_target = ((combined['cpu_percentage'] >= 30) & (combined['cpu_percentage'] <= 60)).sum() / len(combined) * 100
    efficiency.append({'algorithm': alg_name, 'mean_util': mean_util, 'in_target_pct': in_target})

eff_df = pd.DataFrame(efficiency)
x = np.arange(len(eff_df))
ax.bar(x, eff_df['in_target_pct'], color=colors[:len(eff_df)])
ax.set_xticks(x)
ax.set_xticklabels(eff_df['algorithm'], rotation=45, ha='right')
ax.set_ylabel('Time in Target Range (\%)')
ax.set_title('Resource Efficiency: Time Spent in 30-60\% CPU Target')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics table
summary_path = os.path.join(results_dir, 'summary_statistics.csv')
if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
    pivot = summary.pivot(index='metric', columns='algorithm', values='mean')
    display(pivot.style.format('{:.4f}'))
else:
    print('Run benchmark_results.py first to generate summary statistics')